# Partie F — Modélisation prédictive de la réponse à la campagne

Objectif : Prédire la probabilité qu'un client réponde à une campagne marketing.
Variable cible : Response (0/1)
Modèles testés : Régression Logistique, Random Forest, Gradient Boosting, KNN
Critères de choix : performance technique + intérêt métier

In [ ]:
# ── Imports ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, precision_recall_curve,
                             average_precision_score)
from sklearn.model_selection import GridSearchCV
import shap

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

df = pd.read_csv('../data/processed/df_clustered.csv')
print(f"Dataset chargé : {df.shape[0]} lignes x {df.shape[1]} colonnes")
print(f"Taux de reponse global : {df['Response'].mean()*100:.1f}%")

: 

Préparation train/test

In [ ]:
# ── 1. PREPARATION TRAIN / TEST ───────────────────────────
# Variables exclues : Response (cible), Cluster, Segment (labels)
# Education gardée sous forme encodée (Education_Enc)
exclude = ['Response', 'Cluster', 'Segment', 'Education']
features = [c for c in df.columns if c not in exclude]

X = df[features]
y = df['Response']

print(f"Features utilisées : {len(features)}")
print(f"Distribution cible : {y.value_counts().to_dict()}")

# Split stratifié pour conserver le ratio 85/15
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain : {X_train.shape[0]} lignes | Test : {X_test.shape[0]} lignes")
print(f"Taux reponse train : {y_train.mean()*100:.1f}%")
print(f"Taux reponse test  : {y_test.mean()*100:.1f}%")

# Normalisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Sauvegarde scaler
joblib.dump(scaler, '../models/scaler.pkl')
print("\nScaler sauvegardé")

Entraînement des 4 modèles

In [ ]:
# ── 2. ENTRAINEMENT DES MODELES ───────────────────────────
# class_weight='balanced' pour gérer le déséquilibre 85/15
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(
        class_weight='balanced', random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(
        class_weight='balanced', random_state=42, n_estimators=100),
    'Gradient Boosting': GradientBoostingClassifier(
        random_state=42, n_estimators=100),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

results = {}
print("=" * 65)
print("COMPARAISON DES MODELES — CROSS VALIDATION (5 folds)")
print("=" * 65)

for name, model in models.items():
    # Cross-validation AUC-ROC
    auc_scores = cross_val_score(
        model, X_train_scaled, y_train,
        cv=cv, scoring='roc_auc', n_jobs=-1)

    # Cross-validation F1
    f1_scores = cross_val_score(
        model, X_train_scaled, y_train,
        cv=cv, scoring='f1', n_jobs=-1)

    results[name] = {
        'AUC_mean': auc_scores.mean(),
        'AUC_std': auc_scores.std(),
        'F1_mean': f1_scores.mean(),
        'F1_std': f1_scores.std(),
        'model': model
    }

    print(f"\n{name}")
    print(f"   AUC-ROC : {auc_scores.mean():.4f} (+/- {auc_scores.std():.4f})")
    print(f"   F1      : {f1_scores.mean():.4f} (+/- {f1_scores.std():.4f})")

Évaluation sur le test set

In [ ]:
# ── 3. EVALUATION SUR LE TEST SET ────────────────────────
print("=" * 65)
print("EVALUATION SUR LE JEU DE TEST")
print("=" * 65)

test_results = {}
for name, res in results.items():
    model = res['model']
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    auc = roc_auc_score(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)

    test_results[name] = {
        'AUC': auc,
        'AP': ap,
        'y_proba': y_proba,
        'y_pred': y_pred,
        'model': model
    }

    print(f"\n{name}")
    print(f"   AUC-ROC            : {auc:.4f}")
    print(f"   Average Precision  : {ap:.4f}")
    print(classification_report(y_test, y_pred,
          target_names=['Non repondant', 'Repondant']))

Message d'erreur : 
"Échec du démarrage du Kernel. Impossible de démarrer le noyau « venv (Python 3.13.1) » en raison d’un délai d’attente d’utilisation des ports. Pour plus d’informations, consultez Jupyter log."

Solution : on exécute la modélisation via src/modeling.py dans le terminal. Les résultats et modèles sont sauvegardés proprement. On recopie ensuite les outputs dans le notebook pour garder la trace.